In [237]:
%pip install python-dotenv openai numpy pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [238]:
import numpy as np
from dotenv import load_dotenv
from pydantic import BaseModel

from openai import OpenAI

import os
from enum import Enum

In [239]:
load_dotenv()

True

In [240]:
LLM_API_URL = os.getenv("LLM_API_URL")
LLM_API_TOKEN = os.getenv("LLM_API_TOKEN")

print(f"LLM_API_URL: {LLM_API_URL}")
print(f"LLM_API_TOKEN: {LLM_API_TOKEN}")

MODEL = "google/gemma-4-e4b"

LLM_API_URL: http://172.26.128.1:1234/v1
LLM_API_TOKEN: sk-lm-VswvHhzP:xhpy0T4I5gJxaCwNGT44


In [241]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)
'''
response = client.responses.create(
    model=MODEL,
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)'''

'\nresponse = client.responses.create(\n    model=MODEL,\n    instructions="You are a coding assistant that talks like a pirate.",\n    input="How do I check if a Python object is an instance of a class?",\n)\n\nprint(response.output_text)'

# Modélisation du monde

In [ ]:
VOID = 0
PLAYER = 1
ENNEMY = 2
GOLD = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

# Combat deterministe
PLAYER_MAX_HP = 3
ENEMY_MAX_HP = 2

In [ ]:
# Layout: ennemi sur le chemin + piece leurre
# - Joueur (3,0)
# - Ennemi (3,2) bloque le chemin direct vers la piece leurre (3,3)
# - Pieces "sures" plus loin : (0,6) et (6,6)
# => le niveau 'solved' fonce sur le leurre et combat ; 'hint' peut contourner.

initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 3],  # piece sure (0,6)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [1, 0, 2, 3, 0, 0, 0],  # joueur(3,0) ennemi(3,2) leurre(3,3)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],  # piece sure (6,6)
])

initial_map

# Couche de contrat

In [ ]:
H = "HAUT"
B = "BAS"
G = "GAUCHE"
D = "DROITE"

class Direction(str, Enum):
    HAUT = H
    BAS = B
    GAUCHE = G
    DROITE = D

class PlayerDecision(BaseModel):
    #directionJustification: str
    direction : Direction

# Niveau de charge algorithmique -> axe principal du benchmark
class AlgoLevel(str, Enum):
    RAW = "raw"        # LLM raisonne a partir des distances seules
    HINT = "hint"      # deltas signes fournis, le LLM mappe delta -> direction
    SOLVED = "solved"  # regle deterministe explicite, le LLM tamponne

MOVES = {
    H: (-1, 0),
    B: (1, 0),
    G: (0, -1),
    D: (0, 1),
}

# Moteur de perception

In [245]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [246]:
def compute_distances(entities_positions, reference_pos):
    if(len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
    
    return np.round(distances, 2)

In [ ]:
def perception(world_map, player_hp=PLAYER_MAX_HP):
    player_position = localize(world_map, PLAYER)[0]
    gold_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    gold_distances = compute_distances(gold_positions, player_position)
    ennemies_distances = compute_distances(ennemies_positions, player_position)

    # delta signe vers l'entite la plus proche (perception directionnelle)
    def nearest_delta(positions, distances):
        if len(positions) == 0:
            return {"row": 0, "col": 0}
        i = int(np.argmin(distances))
        d_row, d_col = positions[i] - player_position
        return {"row": int(d_row), "col": int(d_col)}

    return {
        "player_hp": int(player_hp),
        "gold_count": len(gold_positions),
        "gold_distances": gold_distances.tolist(),
        "nearest_gold_delta": nearest_delta(gold_positions, gold_distances),
        "ennemies_count": len(ennemies_positions),
        "ennemies_distances": ennemies_distances.tolist(),
        "nearest_enemy_delta": nearest_delta(ennemies_positions, ennemies_distances),
    }

In [248]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('---------------------------------------------------')
    
show_map(initial_map)

·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
---------------------------------------------------


# Moteur de déplacement

In [ ]:
def in_bounds(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos
    return 0 <= r < n_rows and 0 <= c < n_cols

In [ ]:
def move(world_map: np.ndarray, old_pos, new_pos, player_hp, enemy_hp):
    """Resout une action. Combat deterministe : entrer sur un ennemi = attaque,
    -1 PV pour le joueur et -1 PV pour l'ennemi. A 0 PV l'entite meurt."""
    old_pos = (int(old_pos[0]), int(old_pos[1]))
    result = {
        "new_pos": old_pos,
        "player_hp": player_hp,
        "gold_collected": False,
        "combat": False,
        "enemy_killed": False,
        "player_died": False,
        "moved": False,
    }

    if not in_bounds(world_map, new_pos):
        return result  # hors grille -> reste sur place (coup perdu)

    new_pos = (int(new_pos[0]), int(new_pos[1]))
    target = world_map[new_pos]

    if target == ENNEMY:
        result["combat"] = True
        player_hp -= 1
        enemy_hp[new_pos] = enemy_hp.get(new_pos, ENEMY_MAX_HP) - 1
        result["player_hp"] = player_hp

        if player_hp <= 0:
            result["player_died"] = True
            return result

        if enemy_hp[new_pos] <= 0:
            # ennemi tue -> le joueur avance sur la case
            result["enemy_killed"] = True
            del enemy_hp[new_pos]
            world_map[old_pos] = VOID
            world_map[new_pos] = PLAYER
            result["new_pos"] = new_pos
            result["moved"] = True
        # sinon : le joueur attaque mais reste sur place
        return result

    if target in (VOID, GOLD):
        world_map[old_pos] = VOID
        world_map[new_pos] = PLAYER
        result["new_pos"] = new_pos
        result["moved"] = True
        if target == GOLD:
            result["gold_collected"] = True
        return result

    return result  # bloque -> reste sur place

# Moteur de décision

In [ ]:
def decide(player_perception, algo_level=AlgoLevel.HINT) -> PlayerDecision | None:
    gd = player_perception["nearest_gold_delta"]
    ed = player_perception["nearest_enemy_delta"]

    base = f"""
    # Contexte
    - Tu es un joueur sur une grille. Objectif : ramasser le plus d'or possible.
    - Tu as {player_perception['player_hp']} PV. Attaquer un ennemi coute 1 PV.
    - Un ennemi (PV {ENEMY_MAX_HP}) bloque le passage ; le tuer demande {ENEMY_MAX_HP} attaques.

    # Perception
    {player_perception}
    """

    if algo_level == AlgoLevel.RAW:
        guidance = """
    # Consigne
    - Deduis toi-meme la direction vers l'or a partir des distances fournies.
    """
    elif algo_level == AlgoLevel.HINT:
        guidance = f"""
    # Reperes
    - row augmente vers le BAS, diminue vers le HAUT.
    - col augmente vers la DROITE, diminue vers la GAUCHE.
    - Or le plus proche (relatif a toi): row={gd['row']}, col={gd['col']}.
    - Ennemi le plus proche (relatif a toi): row={ed['row']}, col={ed['col']}.

    # Consigne
    - Choisis la direction qui te rapproche de l'or.
    - Evite l'ennemi si tes PV sont bas ; ne le combats que si necessaire.
    """
    else:  # SOLVED
        guidance = f"""
    # Regles deterministes (applique dans l'ordre)
    1. Si row de l'or < 0 -> HAUT
    2. Sinon si row de l'or > 0 -> BAS
    3. Sinon si col de l'or < 0 -> GAUCHE
    4. Sinon si col de l'or > 0 -> DROITE
    - Or le plus proche: row={gd['row']}, col={gd['col']}.
    """

    prompt = base + guidance + "\n    Reponds uniquement avec la direction choisie."

    response = client.beta.chat.completions.parse(
        model = MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format = PlayerDecision,
        temperature = 0,
        seed = 42,
    )

    return response.choices[0].message.parsed or None

# Game loop (simulation)

In [ ]:
def game_loop(world_map: np.ndarray, algo_level=AlgoLevel.HINT, max_turns=20, verbose=True):
    world_map = world_map.copy()
    player_hp = PLAYER_MAX_HP
    enemy_hp = {(int(p[0]), int(p[1])): ENEMY_MAX_HP for p in localize(world_map, ENNEMY)}
    total_gold = len(localize(world_map, GOLD))
    coins_collected = 0
    records = []

    for turn in range(max_turns):
        if len(localize(world_map, GOLD)) == 0:
            break  # tout l'or ramasse

        player_pos = localize(world_map, PLAYER)[0]
        p = perception(world_map, player_hp)

        if verbose:
            print(f"\n===== [Turn {turn + 1}] HP {player_hp} | algo {algo_level.value} =====")
            show_map(world_map)
            print(f"  gold Δ{p['nearest_gold_delta']}  enemy Δ{p['nearest_enemy_delta']}")

        decision = decide(p, algo_level)
        if decision is None:
            break

        d_row, d_col = MOVES[decision.direction.value]
        new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
        res = move(world_map, player_pos, new_pos, player_hp, enemy_hp)
        player_hp = res["player_hp"]
        if res["gold_collected"]:
            coins_collected += 1

        if verbose:
            flags = "".join([
                " [COMBAT]" if res["combat"] else "",
                " [KILL]" if res["enemy_killed"] else "",
                " [GOLD]" if res["gold_collected"] else "",
                " [DEAD]" if res["player_died"] else "",
                " [WASTED]" if (not res["moved"] and not res["combat"]) else "",
            ])
            print(f"  → {decision.direction.value}{flags}")

        records.append({
            "turn": turn + 1,
            "algo_level": algo_level.value,
            "model": MODEL,
            "player_row": int(player_pos[0]),
            "player_col": int(player_pos[1]),
            "player_hp": int(player_hp),
            "gold_delta_row": p["nearest_gold_delta"]["row"],
            "gold_delta_col": p["nearest_gold_delta"]["col"],
            "enemy_delta_row": p["nearest_enemy_delta"]["row"],
            "enemy_delta_col": p["nearest_enemy_delta"]["col"],
            "decision": decision.direction.value,
            "combat": res["combat"],
            "enemy_killed": res["enemy_killed"],
            "gold_collected": res["gold_collected"],
            "coins_collected": coins_collected,
            "coins_remaining": len(localize(world_map, GOLD)),
            "wasted_move": (not res["moved"]) and (not res["combat"]),
            "player_died": res["player_died"],
        })

        if res["player_died"]:
            break

    return records

In [ ]:
records = game_loop(initial_map, algo_level=AlgoLevel.HINT, max_turns=20)

print("\n================ RESUME ================")
print(f"tours joues     : {len(records)}")
print(f"pieces ramassees: {records[-1]['coins_collected'] if records else 0}")
print(f"PV restants     : {records[-1]['player_hp'] if records else PLAYER_MAX_HP}")
print(f"coups perdus    : {sum(r['wasted_move'] for r in records)}")
print(f"combats         : {sum(r['combat'] for r in records)}")

# Todo 01/07